# 7.6 WebAssembly 与浏览器端推理

## 说明

浏览器推理主战场在 **JS/WASM/WebGPU**，本 notebook 用 Python **建模选型与资源预算**；可交互骨架见 [`browser_demo/index.html`](browser_demo/index.html)。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field
from typing import Optional
import time
import hashlib
import hmac
import json
import math

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__}")

## 7.6.1 技术栈对比

In [ ]:
STACKS = [
    ("WebLLM", "TVM WebGPU", "W4A16", "聊天产品原型", "需较新 Chrome/Edge"),
    ("Transformers.js", "ORT Web", "INT8", "HF 生态小模型", "算子覆盖看 ORT"),
    ("MediaPipe LLM", "WebGPU", "W4A16", "Gemma/Phi 官方路径", "Google 栈"),
    ("llama.cpp WASM", "SIMD WASM", "Q4_K_M", "CPU 兜底/兼容", "慢于 WebGPU"),
    ("Chrome Prompt API", "内置 Gemini Nano", "系统管理", "零下载", "仅支持的 Chrome 版本"),
]
print(f"{'框架':18s} {'引擎':14s} {'量化':8s} {'适合':14s} 限制")
for r in STACKS:
    print(f"{r[0]:18s} {r[1]:14s} {r[2]:8s} {r[3]:14s} {r[4]}")

## 7.6.2 浏览器内存与加载预算

In [ ]:
def browser_fit(model_mb: float, tab_limit_mb: float = 1536, runtime_mb: float = 120, kv_mb: float = 200) -> dict:
    need = model_mb + runtime_mb + kv_mb
    return {
        "need_mb": need,
        "ok": need < tab_limit_mb,
        "headroom_mb": tab_limit_mb - need,
        "hint": "可跑" if need < tab_limit_mb else "需更小量化/更短上下文/分片加载",
    }


for name, mb in [("Llama-3.2-1B Q4", 700), ("Gemma-2B W4", 1200), ("Phi-3-mini W4", 2100)]:
    print(name, browser_fit(mb))

## 7.6.3 WASM vs WebGPU 延迟模型

In [ ]:
def estimate_tok_s(backend: str, model_b: float, quant_bits: int) -> float:
    # 极简相对模型：WebGPU >> WASM；参数越大越慢；比特越低略快（带宽）
    base = {"webgpu": 28.0, "wasm": 12.0, "prompt_api": 35.0}[backend]
    return base * (1.5 / model_b) * (4 / quant_bits) * 1.2


print("=== 相对吞吐估算 (tok/s) ===")
for backend in ("webgpu", "wasm", "prompt_api"):
    for bits in (4, 8):
        print(f"{backend:10s} W{bits}: {estimate_tok_s(backend, 1.5, bits):5.1f}")

## 7.6.4 何时选浏览器端？

In [ ]:
def choose_runtime(install_ok: bool, need_offline_app: bool, zero_install: bool, model_mb: float) -> str:
    if zero_install and model_mb <= 1200:
        return "WebLLM / Transformers.js（CDN + Service Worker 缓存）"
    if not install_ok and zero_install:
        return "Chrome Prompt API（若可用）或云端"
    if need_offline_app:
        return "原生 App（ExecuTorch / llama.cpp / CoreAI）"
    return "Ollama / 本地原生服务 + Web 前端"


cases = [
    dict(install_ok=False, need_offline_app=False, zero_install=True, model_mb=900),
    dict(install_ok=True, need_offline_app=True, zero_install=False, model_mb=1800),
    dict(install_ok=False, need_offline_app=False, zero_install=True, model_mb=2500),
]
for c in cases:
    print(c, "->", choose_runtime(**c))

## 小结

- 浏览器适合：**零安装试用、隐私演示、轻量助手**。
- 生产级手机助手仍以原生运行时为主。
- 打开 `browser_demo/index.html` 查看接入骨架（不捆绑大模型权重）。